# ACC 380K — Case 1
## Does the foot traffic support the revenue? Texas Roadhouse, 2017–2026

**Assignment version 0.3.10 · October 26, 2026**  
**Notebook due:** Thursday, November 5, 2026, 11:59 PM CT.  
**Extra-credit revenue guess due:** Thursday, November 5, 2026, 12:00 PM (noon) CT, in a **separate** Canvas assignment.

Forming an independent prediction of a number like future revenue is a valuable skill in accounting. Buyers and their advisers do it in due diligence for mergers and acquisitions, analysts do it in investment research, and external auditors do it when they form independent expectations for analytical procedures.

In this case you do it for **Texas Roadhouse, Inc.** (Nasdaq: TXRH), which runs Texas Roadhouse, Bubba's 33, and Jaggers restaurants. You have about 300,000 weekly foot-traffic estimates for every US Texas Roadhouse restaurant from January 2017 to September 2026, and the quarterly revenue the company reported to the SEC. You will put visits next to revenue, decide how far the two move together, explain where and why they part, and predict Q3 2026 revenue before the company reports it.

How you build the analysis is up to you. The **Deliverables** section says what to produce and answer; optional hints follow it.

### Timeline

- **By noon (12:00 PM CT) Thursday, November 5:** rough out your estimate: get your analysis working far enough to produce `q3_2026_prediction_musd`, and enter that number in the separate extra-credit assignment. The estimate does not need to be polished yet. Texas Roadhouse reports after the market closes that day.
- **By 11:59 PM CT that night (Thursday, November 5):** finish your code comments and documentation, the charts, and the written answers, and submit the notebook. Your final notebook must still compute the same number you entered for extra credit (see Deliverable 3).

### Before you begin

1. Choose **File → Save a copy in Drive** and name your copy `lastname_firstname_case1.ipynb`.
2. Read the whole assignment before you write code.
3. This is an **individual** assignment. AI assistants, course notes, documentation, and ordinary internet resources are allowed (see "Working with AI"), but written answers must be your own words and cite numbers from your own output.

### Advice for Working with AI

You are encouraged to use AI (for example, Gemini in Colab) to write this code. You are responsible for it: **read every line, understand what it does, and confirm it gives correct answers.**

**If you don't understand the AI's code, you have two options.** Don't leave a line in your notebook that you can't explain.

1. **Ask the AI to teach you.** Have it explain the code line by line until you could explain it yourself. For example: "Explain what this line does, step by step, as if I've only learned loops and pandas masks."
2. **Ask the AI to rewrite it with tools you know.** Work with it to replace the code with an approach you already understand. For example: "Rewrite this using only a for loop and pd.concat, which I already know."

**Steer the tools.** Left alone, AI often picks tools we haven't covered. Ask it to use the tools you already know, so you can audit what it writes. For example:

- "Use `groupby`, `.agg`, `merge`, and boolean masks. Don't use `resample` or `apply` with lambdas."
- "Use `.pct_change(4, fill_method=None)` for year-over-year growth."
- "Don't fill or drop missing values without telling me."

**Two ways to audit.** *Auditing the code* means reading each line and knowing what it does. *Auditing the output* means checking the results for problems: is there one row per quarter, does each quarter's visit measure make sense, does 2026Q3 have visits but no revenue, are there growth rates where there shouldn't be? Use both.

**Your writing must match your output.** After your final restart and run all, check that every number in your written answers matches what the code now prints. AI cannot do this for you. Any number or claim in your written answers should match what your notebook prints; don't copy an AI's description without checking it against your results.

## The data

Paste the **class code** from Canvas between the quotes in `CLASS_KEY`, then run the two cells below. Do not share the code, and do not post the files or a notebook that shows their rows anywhere public.

| File | One row is |
|---|---|
| `TXRH_case1_visits.csv` (about 43 MB) | one Texas Roadhouse restaurant in one week (Monday–Sunday) |
| `TXRH_quarterly_revenue.csv` | one fiscal quarter |

**Columns in `TXRH_case1_visits.csv`**

| Column | What it is |
|---|---|
| `PERSISTENT_ID_STORE` | An ID for the restaurant that stays the same from week to week. |
| `REGION` | The state the restaurant is in (for example `TX`). |
| `DATE_RANGE_START` | The Monday the week starts on, as text like `07/06/2026`. |
| `VISIT_COUNTS` | Estimated visits to the restaurant that week. One person who comes twice counts twice. |
| `VISITOR_COUNTS` | Estimated number of different people who visited that week. One person who comes twice counts once, so this is never more than `VISIT_COUNTS`. |
| `MEDIAN_DWELL` | The median length of a visit that week, in minutes. Blank for a few restaurant-weeks. |
| `IOS_DEVICES` | Number of iPhones (and other Apple devices) in the sample seen at the restaurant that week. Blank for a few restaurant-weeks. |
| `ANDROID_DEVICES` | Number of Android phones in the sample seen at the restaurant that week. Blank for a few restaurant-weeks. |
| `MON_VISITS` … `SUN_VISITS` | Seven columns (`MON_VISITS`, `TUE_VISITS`, `WED_VISITS`, `THU_VISITS`, `FRI_VISITS`, `SAT_VISITS`, `SUN_VISITS`): estimated visits on each day of that week, Monday first. They add up to `VISIT_COUNTS`, give or take rounding. |
| `DISTANCE_FROM_HOME` | The median distance visitors traveled from home, in meters. Blank for a few restaurant-weeks. |
| `STREET_ADDRESS` | The restaurant's street address. |
| `CITY` | The restaurant's city. |
| `POSTAL_CODE` | The restaurant's ZIP code, stored as text so leading zeros are kept (for example `05495`); a few include the ZIP+4 suffix (for example `28546-5503`). `pd.read_csv` reads it as a number and drops leading zeros unless you add `dtype={"POSTAL_CODE": str}`. |
| `LATITUDE` | The restaurant's latitude, in decimal degrees. |
| `LONGITUDE` | The restaurant's longitude, in decimal degrees (negative = west). |

The device columns count phones in the data provider's sample, not all customers, so they are much smaller than the visit estimates.

**Columns in `TXRH_quarterly_revenue.csv`:** `quarter` (like `2025Q3`), `fiscal_year`, `fiscal_quarter`, `period_start`, `period_end` (text like `2025-07-02`), `days`, `revenue_usd`, `source`.

**Optional:** you may use any of these columns in your analysis or your prediction. You don't have to; the core deliverables need only `VISIT_COUNTS` and the restaurant count.

**Caveats.**

- **Visits are estimates.** They are Advan (formerly SafeGraph) estimates from cell-phone location data, downloaded from Dewey, not receipt counts. The number of restaurants the file follows grows over time (about 490 at the start, about 690 at the end).
- **The data provider switched vendors in January 2023.** The foot-traffic provider switched the vendor that supplies its underlying data in January 2023. The switch may have changed some things about how the data are collected and compiled. Written question 4.2 (in section 4 of Your work) asks whether you see any evidence of it.
- **Revenue is total company revenue** from the 10-Q and 10-K filings, so it includes Bubba's 33, Jaggers, international, and franchise revenue that the visit file does not cover.
- **Q4 is derived.** The fourth quarter is not filed on its own, so it is the annual figure minus the first three quarters.
- **Fiscal quarters are not calendar quarters.** Texas Roadhouse's fiscal year ends on the last Tuesday of December; its quarters run Wednesday to Tuesday, 13 weeks each, with a 14th week in the fourth quarter of a 53-week year (fiscal 2019 and 2024).
- **2026Q3 revenue is blank.** The company reports it on **Thursday, November 5, 2026**, after the market closes. The visit file already covers every full week of the quarter (the last is September 21–27).
- **Coverage gaps at the ends.** The visit data do not cover the last two days of 2026Q3 (September 28–29, 2026) or the first five days of 2017Q1 (December 28, 2016–January 1, 2017).

A wrong or missing code gets **403**, and `curl` saves the text `Forbidden: check the class code from Canvas.` in place of the data. Fix the code and rerun the download cell. If the runtime is deleted, run these cells again.

In [ ]:
CLASS_KEY = ""  # paste the class code from Canvas
!curl -H "X-Class-Key: {CLASS_KEY}" -o TXRH_case1_visits.csv https://acc380k-data.nicholashallman.workers.dev/data/TXRH_case1_visits.csv
!curl -H "X-Class-Key: {CLASS_KEY}" -o TXRH_quarterly_revenue.csv https://acc380k-data.nicholashallman.workers.dev/data/TXRH_quarterly_revenue.csv

import pandas as pd
visits = pd.read_csv("TXRH_case1_visits.csv", dtype={"POSTAL_CODE": str})
revenue = pd.read_csv("TXRH_quarterly_revenue.csv")
visits

In [ ]:
revenue

## Deliverables

Your finished notebook must contain everything below. Put each item in its numbered section under **Your work**; the sections match this list. How you organize your tables, what you name them, and how you build them is up to you; the only required name is `q3_2026_prediction_musd`.

### What applies to every deliverable (graded)

- **Comment your code clearly.** Explain what each step does and why, using `#` comments in code cells, short text cells between code cells, or (most likely) a combination of both. A reader who did not write the code should be able to follow it. **Comments and documentation are graded.**
- **The notebook must run.** **Runtime → Restart session and run all** must finish with no errors and reproduce every table, chart, and number you cite.
- **Outside data must be runnable by your instructor.** If you use any data that is not in the two class files (for example, management's comments, analyst figures, or another data set), include it in a way your instructor can run when grading. Any of these is fine:
  - load it from a public URL in a code cell;
  - type the numbers into a code cell, with a comment saying where they came from;
  - upload the file(s) to the Case 1 Canvas assignment along with your notebook, and load them by that exact file name (for example `pd.read_csv("txrh_comps.csv")`).

  Whichever you choose, explain in a text cell where you got the data and how. Don't load a file from a location only you can reach (for example, your own Google Drive).

### 1. Analysis

1. **Weekly visits and restaurant counts.** A table with one row per week showing total visits across all restaurants and the number of restaurants in the file that week. *Complete:* all 508 weeks are present, each week appears once, and you show enough of the table (for example its first and last rows and its length) for a reader to check it.
2. **Quarterly visits lined up with revenue.** A table with one row for each fiscal quarter in `revenue` (39 rows, 2017Q1 through 2026Q3) showing: a measure of visits for that quarter (your *visit measure*: a single quarterly number you build from `VISIT_COUNTS`; you decide exactly how, and say what it is), that quarter's reported revenue, year-over-year growth in revenue, year-over-year growth in visits, and the gap (revenue growth minus visit growth).

   Fiscal quarters and the Monday–Sunday visit weeks do not line up, so some weeks straddle two quarters, and quarters differ in length. How you handle that is your choice; explain the choice you made in written question 4.3 ("Lining up weeks with quarters"), in section 4 of Your work.

   *Complete:* every quarter appears once; 2026Q3 has a visit measure but its revenue, revenue growth, and gap are blank (not filled in); and the growth rates compare each quarter with the same quarter a year earlier.
3. **A backtest.** Test at least **two** prediction methods of your choice on the known quarters 2024Q4, 2025Q1, 2025Q2, and 2025Q3. Each prediction may use only information available before that quarter's revenue was reported (revenue through the prior quarter, visits through the quarter itself). Describe each method in a comment or text cell clearly enough that a reader could reproduce it. *Complete:* a table showing, for each method and each of the four quarters, the predicted revenue, the actual revenue, and the percentage error `(predicted - actual) / actual`, plus two averages for each method: the **mean signed error** (the plain average of the four percentage errors, which shows whether the method tends to over- or underpredict) and the **mean absolute error** (the average of the errors ignoring their sign, which shows how far off the method typically is).

   *One idea (not required):* assume revenue grows like visits plus a typical difference between the two growth rates, for example last year's same-quarter revenue × (1 + this quarter's visit growth + the average gap between revenue growth and visit growth in recent quarters). There are several reasonable ways to measure and apply that gap; if you use this idea, say which one you chose.

### 2. Charts

1. Weekly total visits over time.
2. Restaurants per week over time.
3. Year-over-year revenue growth and visit growth together on one chart, by quarter.

*Complete:* each chart has a title, labeled axes with units, and a legend (or labels) that tells a reader which line is which.

### 3. Prediction

Set **`q3_2026_prediction_musd`** to your prediction of Texas Roadhouse's **total Q3 2026 revenue in millions of dollars** (for example `1550.0`). Any method is fine, including one from your backtest, and you may use public information released before the extra-credit deadline (noon CT, November 5) (for example, management's comments on comparable restaurant sales) or any of the extra foot-traffic columns.

- **Computed, not typed.** `q3_2026_prediction_musd` must be **calculated by your notebook's code** from the data and any outside inputs, not typed in as a number. Print it at the end of this section.
- **It must support your extra-credit guess.** The number your notebook computes should match the number you entered in the extra-credit assignment. If you refine your method after you submit your guess, keep the method that produced your extra-credit guess in the notebook (you may show the refined one too, clearly labeled).
- **Document it.** In a text cell in this section, say in a few sentences how the prediction is built (the steps and inputs; written question 4.5 asks you to evaluate it) and list any outside data you used and where it came from.

**The extra-credit guess is submitted separately.** Enter your number in the Canvas assignment "Case 1 Extra Credit: Q3 2026 Revenue Guess" by **12:00 PM (noon) CT on Thursday, November 5, 2026**, before Texas Roadhouse reports after the market closes that day. Only the Canvas entry counts; a number that appears only in this notebook is not eligible. The three predictions closest to the reported number in each section earn extra credit. The rest of the notebook is due **11:59 PM CT that night**.

### 4. Written analysis

Answer in the text cells in section 4 of Your work, in your own words. Cite specific quarters, weeks, and numbers from your own output; an answer that makes claims without numbers from your notebook is incomplete.

1. **The data's limits** (3–5 sentences). What happens to visits in 2020–2021, and to the number of restaurants in the file over time? Which parts of reported revenue does the visit file not cover? What does this mean for which quarters you can compare?
2. **The January 2023 data-vendor switch** (3–5 sentences). As described in "The data," the foot-traffic provider switched data vendors in January 2023. Do you see any evidence of that switch in the data? If so, what is it (cite weeks, quarters, or numbers); if not, what did you check?
3. **Lining up weeks with quarters** (3–5 sentences). How did you handle weeks that straddle two fiscal quarters, and quarters of different lengths, and why?
4. **Do visits support revenue?** (about 250 words). From 2023 through Q2 2026, does revenue growth move with visit growth, and is the gap steady? Give at least three business reasons revenue could grow faster than visits and say which you could verify in the 10-K. If one quarter's gap jumped well above the usual, what would you look into next, and what information would you ask the company for? Could counting how often certain words or phrases (for example "menu price" or "comparable restaurant sales") appear in Texas Roadhouse's 10-K and 10-Q filings over time help you fine-tune your prediction? If so, how? (You are not expected to actually do this, just explain how you might do it.)
5. **Evaluating your prediction method** (3–5 sentences). How far off was each backtested method, on average and in which direction? Why did you choose the method you used for `q3_2026_prediction_musd` over the alternatives? What could make Q3 2026 different from the quarters you tested? (Describe *how* the prediction is built in the text cell in section 3; here, evaluate it.)

## Optional hints

You do not have to follow these. Some suggestions involve commands we have not covered in class, which you should explore yourself should you choose to use them.

- **Dates.** `pd.to_datetime(visits["DATE_RANGE_START"], format="%m/%d/%Y")` turns the month-first text into dates. `period_start` and `period_end` are year first and convert with plain `pd.to_datetime`.
- **One row per week.** `groupby` on the week with `.agg(...)` can give the visit total (`"sum"`) and the restaurant count (`"size"`) at once; finish with `.reset_index()`.
- **Matching weeks to fiscal quarters.** `pd.merge_asof` matches each row on the left to the last row on the right whose key is at or before it, so it can match each week to the quarter that started on or before a date you choose. A loop over the rows of `revenue` with a boolean mask also works. Either way, decide what to do with weeks that straddle a quarter boundary.
- **Quarters of different lengths.** Most fiscal quarters are 91 days, but two are 98. Think about whether your quarterly visit measure is comparable across quarters of different lengths and across years.
- **Joining.** `revenue.merge(your_quarterly_table, on="quarter", how="left", validate="one_to_one")` keeps every revenue quarter, including 2026Q3. Sort by `period_start` afterward so "four rows back" means "a year earlier."
- **Year-over-year growth.** Restaurants are seasonal, so compare each quarter with the same quarter a year earlier, four rows back: `.pct_change(4, fill_method=None)`. Write `fill_method=None`: 2026Q3 revenue is blank, and without it pandas quietly fills the blank with the previous quarter's revenue and reports a made-up 2026Q3 growth rate (newer pandas also prints a FutureWarning).
- **The changing restaurant count.** The file follows more restaurants over time, so growth in visits partly reflects coverage, not just traffic. Be careful with growth rates for 2020 through early 2022 (COVID).
- **The blank Q3.** 2026Q3 has visits but no revenue. Make sure nothing in your code fills, drops, or averages that blank by accident.
- **Other prediction methods.** Ideas: last year's revenue per visit (adjusted for its trend) × this quarter's estimated visits, or a simple OLS regression of revenue growth on visit growth. pandas itself has no regression function, but the `statsmodels` package (already installed in Colab) does: `import statsmodels.formula.api as smf`, then `model = smf.ols("rev_growth ~ visit_growth", data=your_table).fit()` and `model.params` gives the intercept and slope (`model.summary()` shows the familiar regression table). Drop rows with blank growth first, and fit only on quarters before the one you are predicting.
- **The extra columns.** Ratios are often easier to compare over time than raw counts: for example `VISIT_COUNTS / VISITOR_COUNTS` (visits per visitor), `IOS_DEVICES / (IOS_DEVICES + ANDROID_DEVICES)` (iPhone share), or `(FRI_VISITS + SAT_VISITS + SUN_VISITS) / VISIT_COUNTS` (weekend share). A few rows have blanks in these columns.

## Your work

Each numbered section below matches a deliverable. Put your work for each item in its section, and add as many code and text cells inside a section as you need (**Insert → Code cell / Text cell**). Keep everything in order so the notebook runs top to bottom, and comment as you go.

## 1. Analysis

### 1.1. Weekly visits and restaurant counts

*Deliverable:* One row per week: total visits across all restaurants and the number of restaurants in the file that week.

### 1.2. Quarterly visits lined up with revenue

*Deliverable:* One row per fiscal quarter (2017Q1–2026Q3): a visit measure, revenue, YoY revenue growth, YoY visit growth, and the gap. 2026Q3 revenue stays blank.

### 1.3. Backtest

*Deliverable:* At least two methods of your choice on 2024Q4–2025Q3: predicted, actual, and percentage error for each quarter, plus each method's mean signed error and mean absolute error.

## 2. Charts

### 2.1. Chart: weekly total visits

*Deliverable:* Weekly total visits over time, with a title and labeled axes.

### 2.2. Chart: restaurants per week

*Deliverable:* Restaurants per week over time, with a title and labeled axes.

### 2.3. Chart: revenue growth vs. visit growth

*Deliverable:* Year-over-year revenue growth and visit growth on one chart, by quarter, with a legend.

## 3. Prediction

*Deliverable:* Compute `q3_2026_prediction_musd` (total Q3 2026 revenue, $ millions) with code, print it, and describe how it is built (steps and inputs) and list any outside data with sources in the text cell.

In [ ]:
q3_2026_prediction_musd = None  # replace with a calculation, not a typed number
print(q3_2026_prediction_musd)

**How my prediction is built (steps and inputs), and outside data used (with sources):**

_Your answer here._

## 4. Written analysis

Answer in your own words and cite quarters, weeks, and numbers from your own output.

### 4.1. The data's limits

_Your answer here._

### 4.2. The January 2023 data-vendor switch

_Your answer here._

### 4.3. Lining up weeks with quarters

_Your answer here._

### 4.4. Do visits support revenue?

_Your answer here._

### 4.5. Evaluating your prediction method

_Your answer here._

## Before submitting

- **Runtime → Restart session and run all** finishes with no errors.
- Every numbered section has your work, and your code is clearly commented (`#` comments and/or text cells).
- `q3_2026_prediction_musd` is computed by your code (not typed in), printed in section 3, and matches your extra-credit guess.
- Any outside data loads from a public URL, a code cell, or a file you upload to Canvas with your notebook, and section 3 says where it came from.
- The three charts appear and are labeled.
- Every number in your written answers matches your output after the final restart and run all.
- Optional: your extra-credit guess is entered in the separate Canvas assignment by **noon CT, Thursday, November 5**.
- The notebook is saved as `lastname_firstname_case1.ipynb` and submitted to the Case 1 assignment on Canvas by **11:59 PM CT, Thursday, November 5**, with any outside data files you load.